In [20]:
import pandas as pd, os, shutil, numpy as np, sys, time
from collections import defaultdict

In [ ]:
"""注：在手动替换步骤完毕之后再运行此文件！"""
print()
confirm = input('确认已手动替换完毕？(输入任意字符继续...):')
print()
if not confirm:
    print()
    print('用户未确认')
    print()
    time.sleep(3)
    sys.exit(0)


In [29]:
yr = 2024

In [3]:
cate_err = pd.read_excel('../输出文件/其他/错误.xlsx',sheet_name='策略错误')
name_err = pd.read_excel('../输出文件/其他/错误.xlsx',sheet_name='名称错误')

In [4]:
cate_err.info()
name_err.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 636 entries, 0 to 635
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   公司名称          636 non-null    object
 1   产品全称          633 non-null    object
 2   投资类型          631 non-null    object
 3   投资策略          597 non-null    object
 4   投资类型_amended  132 non-null    object
 5   投资策略_amended  476 non-null    object
 6   投资策略文字描述      459 non-null    object
 7   处理方式          5 non-null      object
dtypes: object(8)
memory usage: 39.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 189 entries, 0 to 188
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   公司名称          189 non-null    object
 1   表名            189 non-null    object
 2   产品全称          189 non-null    object
 3   产品全称_amended  11 non-null     object
dtypes: object(4)
memory usage: 6.0+ KB


In [8]:
a = cate_err.copy()
b = name_err.copy()

In [11]:
a = a.drop(a[(a['投资类型_amended'].isnull())&(a['投资策略_amended'].isnull())].index)

In [12]:
a.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 494 entries, 1 to 635
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   公司名称          494 non-null    object
 1   产品全称          494 non-null    object
 2   投资类型          494 non-null    object
 3   投资策略          460 non-null    object
 4   投资类型_amended  132 non-null    object
 5   投资策略_amended  476 non-null    object
 6   投资策略文字描述      336 non-null    object
 7   处理方式          0 non-null      object
dtypes: object(8)
memory usage: 34.7+ KB


In [13]:
b = b.drop(b[b['产品全称_amended'].isnull()].index)

In [14]:
b.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 11 entries, 0 to 73
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   公司名称          11 non-null     object
 1   表名            11 non-null     object
 2   产品全称          11 non-null     object
 3   产品全称_amended  11 non-null     object
dtypes: object(4)
memory usage: 440.0+ bytes


In [15]:
company_setA = set(a['公司名称'].unique())
company_setB = set(b['公司名称'].unique())

In [17]:
company_setC = company_setA | company_setB

In [18]:
company_setC

{'上海东方',
 '上海海通',
 '东方财富',
 '中信证券',
 '信达证券',
 '华金证券',
 '国海证券',
 '国联证券',
 '国金证券',
 '天风证券',
 '太平洋证券',
 '安信证券',
 '平安证券',
 '广发资管',
 '招商证券',
 '浙商证券',
 '申万宏源',
 '申港证券',
 '第一创业',
 '财达证券',
 '银河金汇',
 '首创证券'}

In [32]:
sheetname = ['基础信息', '产品净值', '历任基金经理信息', '规模', '拆分', '分红']
test = '测试版'

In [31]:
for company_name in list(company_setC):
    temp_a = a[a['公司名称']==company_name]
    temp_b = b[b['公司名称']==company_name]
    xls = pd.ExcelFile(f'../数据源{test}/{yr}证券公司集合资管计划金牛奖机构填报数据表-{company_name}.xlsx')
    sheet_dict = defaultdict(list)
    for sheet_name in sheetname:
        df = pd.read_excel(xls,sheet_name=sheet_name)
        sheet_dict[sheet_name].append(df)
    
    if not temp_a.empty:
        temp_info = sheet_dict['基础信息'][-1].copy()
        temp_info = temp_info[temp_info['投资类型'].notna()]
        for index,row in temp_a.iterrows():
            prod_name = row['产品全称']
            prod_type = row['投资类型_amended']
            prod_strat = row['投资策略_amended']
            temp_info.loc[temp_info['产品全称']==prod_name, '投资类型'] = prod_type
            temp_info.loc[temp_info['产品全称']==prod_name, '投资策略'] = prod_strat
        df = temp_info.copy()
        sheet_dict['基础信息'].append(df)
    
    if not temp_b.empty:
        for st in temp_b['表名'].unique():
            if st == '基础信息':
                for index,row in temp_b[temp_b['表名']==st].iterrows():
                    prod_name = row['产品全称']
                    prod_name_amended == row['产品全称_amended']
                    temp_info.loc[temp_info['产品全称']==prod_name,'产品全称'] = prod_name_amended
                df = temp_info.copy()
                sheet_dict[st].append(df)
            else:
                temp = sheet_dict[st][-1].copy()
                for index,row in temp_b[temp_b['表名']==st].iterrows():
                    prod_name = row['产品全称']
                    prod_name_amended = row['产品全称_amended']
                    temp.loc[temp['产品全称']== prod_name,'产品全称'] = prod_name_amended
                
                df = temp.copy()
                sheet_dict[st].append(df)
    with pd.ExcelWriter(f'../数据源{test}/{yr}证券公司集合资管计划金牛奖机构填报数据表-{company_name}.xlsx') as writer:
        for sheet, dfs in sheet_dict.items():
            df = dfs[-1]
            df.to_excel(writer,sheet_name=sheet,index=False)
                
                
        